# OULAD FE — Sequence Version (Realtime EWS) cho ML

**Mục tiêu:** giữ nguyên các feature đã thiết kế trong notebook tabular giữa kỳ,
nhưng tái cấu trúc thành dạng chuỗi theo tuần (causal per-week)
để tạo Snapshots dùng cho các mô hình Cây (XGBoost, LightGBM).

**Nguyên tắc:** mỗi feature `f` trong bản tabular → trở thành chuỗi `f(t)` với `t = 0..T_max-1`,
trong đó `f(t)` chỉ dùng data trong `[day 0, day 7*(t+1)-1]` (không leak future).

**Granularity:** Weekly, T_max = ceil(CUTOFF_DAY / 7) = 25 tuần (với CUTOFF_DAY=170).


## 0. Setup & Load Data

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from tqdm import tqdm
warnings.filterwarnings('ignore')

CUTOFF_DAY = 170
RANDOM_SEED = 42
WEEK_LEN = 7
T_MAX = int(np.ceil(CUTOFF_DAY / WEEK_LEN))  # 25 weeks
print(f'T_MAX (so tuan): {T_MAX}')

INPUT_PATH = '/kaggle/input/datasets/danielndap/ouladdata/'  # Ban co the can them ten dataset vao sau, VD: '/kaggle/input/oulad/'
OUTPUT_PATH = '/kaggle/working/seq_data/'
df_assessments         = pd.read_csv(INPUT_PATH + 'assessments.csv')
df_studentInfo         = pd.read_csv(INPUT_PATH + 'studentInfo.csv')
df_studentAssessment   = pd.read_csv(INPUT_PATH + 'studentAssessment.csv')
df_studentRegistration = pd.read_csv(INPUT_PATH + 'studentRegistration.csv')
df_studentVle          = pd.read_csv(INPUT_PATH + 'studentVle.csv')
df_vle                 = pd.read_csv(INPUT_PATH + 'vle.csv')

# Binary target: 1 = Fail/Withdrawn (can can thiep), 0 = Pass/Distinction
target_map = {'Withdrawn': 1, 'Fail': 1, 'Pass': 0, 'Distinction': 0}
df_studentInfo['target'] = df_studentInfo['final_result'].map(target_map)
print(df_studentInfo['target'].value_counts())

T_MAX (so tuan): 25
target
1    17208
0    15385
Name: count, dtype: int64


In [2]:
# Cleaning giong notebook goc
df_studentRegistration['date_registration'] = pd.to_numeric(
    df_studentRegistration['date_registration'], errors='coerce')
df_studentRegistration['date_registration'] = df_studentRegistration.groupby(
    ['code_module','code_presentation'])['date_registration'].transform(lambda x: x.fillna(x.median()))
if 'date_unregistration' in df_studentRegistration.columns:
    df_studentRegistration = df_studentRegistration.drop(columns=['date_unregistration'])

df_studentInfo['imd_band'] = df_studentInfo['imd_band'].fillna('Unknown')
df_studentInfo['age_band'] = df_studentInfo['age_band'].fillna('Unknown')

if ('week_from' in df_vle.columns) and ('week_to' in df_vle.columns):
    df_vle = df_vle.drop(columns=['week_from','week_to'])

df_studentAssessment['score'] = df_studentAssessment['score'].fillna(0)
print('Tien xu ly xong.')

Tien xu ly xong.


## 1. Static Features (khong doi theo thoi gian)

Day la cac feature da co trong tabular version, **khong thay doi theo tuan**, se broadcast vao moi timestep:
- Demographics: `gender`, `age_band`, `imd_band`, `region`, `highest_education`, `disability`, `studied_credits`, `num_of_prev_attempts`
- Registration: `date_registration`
- Pre-course engagement: `pre_course_clicks`, `pre_course_active_days`, `has_pre_course_activity`
- Context: `code_module`, `code_presentation`, `module_semester`, `semester`

In [3]:
# 1.1 Pre-course engagement (giu nguyen logic notebook goc)
pre_courses_vle = df_studentVle[df_studentVle['date'] < 0].copy()
pre_courses_engagement = pre_courses_vle.groupby(['id_student']).agg(
    pre_course_clicks      = ('sum_click', 'sum'),
    pre_course_active_days = ('date', 'nunique')
).reset_index()

df_studentInfo = df_studentInfo.drop(columns=['pre_course_clicks','pre_course_active_days'], errors='ignore')
df_studentInfo = df_studentInfo.merge(pre_courses_engagement, on='id_student', how='left')
df_studentInfo[['pre_course_clicks','pre_course_active_days']] = \
    df_studentInfo[['pre_course_clicks','pre_course_active_days']].fillna(0)
df_studentInfo['has_pre_course_activity'] = (df_studentInfo['pre_course_active_days'] > 0).astype(int)
print('Pre-course done.')

Pre-course done.


In [4]:
# 1.2 Merge registration + context features
df_static = df_studentInfo.copy()
df_static = df_static.merge(
    df_studentRegistration[['code_module','code_presentation','id_student','date_registration']],
    on=['code_module','code_presentation','id_student'], how='left'
)
df_static['module_semester'] = df_static['code_module'] + '_' + df_static['code_presentation']
df_static['semester']        = df_static['code_presentation'].str[-1]
print('Static shape:', df_static.shape)
df_static.head(3)

Static shape: (32593, 19)


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,disability,final_result,target,pre_course_clicks,pre_course_active_days,has_pre_course_activity,date_registration,module_semester,semester
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,N,Pass,0,98.0,1.0,1,-159.0,AAA_2013J,J
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,N,Pass,0,215.0,7.0,1,-53.0,AAA_2013J,J
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,Y,Withdrawn,1,102.0,6.0,1,-92.0,AAA_2013J,J


## 2. Dynamic (Per-Week) Features — Causal

Day la phan CORE cua ban sequence. Moi feature dynamic tai tuan `t` chi dung data trong `[0, 7*(t+1)-1]`.

**Mapping tu tabular -> sequence:**

| Feature tabular (bản Tabular) | Feature sequence (bản ML Snapshots)                         |
|--------------------------|----------------------------------------------------|
| `total_clicks`           | `clicks_this_week_t`, `cum_clicks_t`               |
| `active_days`            | `active_days_this_week_t`, `cum_active_days_t`     |
| `max_daily_clicks`       | `max_daily_clicks_t` (trong tuan t)                |
| `median_daily_clicks`    | `median_daily_clicks_t`                            |
| `std_clicks`             | `std_clicks_t`                                     |
| `clicks_per_active_day`  | `clicks_per_active_day_t`                          |
| `active_rate`            | `active_rate_so_far_t`                             |
| `days_since_last_active` | `days_since_last_active_t` (tai cuoi tuan t)       |
| `active_span`            | `active_span_t`                                    |
| `click_slope`            | `click_slope_recent_t` (slope cua tuan [t-3, t])   |
| `click_slope_r2`         | `click_slope_r2_recent_t`                          |
| `clicks_{oucontent,...}` | `clicks_{type}_this_week_t`, `ratio_{type}_t`      |
| `weekend_clicks`         | `weekend_clicks_this_week_t`, `weekend_ratio_t`    |
| `activity_density`       | `activity_density_so_far_t`                        |
| `gap_trend`, `gap_std`   | `gap_trend_t`, `gap_std_t` (causal)                |
| `weighted_score_before_cutoff` | `weighted_score_so_far_t`                    |
| `tma/cma_submission_rate`| `tma/cma_submission_rate_so_far_t`                 |
| `avg_days_before_deadline` | `avg_days_before_deadline_so_far_t`              |
| `n_missing_submission`   | `n_missing_so_far_t`                               |
| `mean_score`             | `mean_score_so_far_t`                              |
| `total_assessments`      | `n_assessments_submitted_so_far_t`                 |

In [5]:
# 2.1 Chuan bi panel: them cot week cho studentVle va studentAssessment

# Filter chi lay data trong khoang [0, CUTOFF_DAY]
vle = df_studentVle[
    (df_studentVle['date'] >= 0) & (df_studentVle['date'] <= CUTOFF_DAY)
].copy()
vle['week'] = vle['date'] // WEEK_LEN  # week 0 = day 0..6
vle['day_of_week'] = vle['date'] % WEEK_LEN  # 0=Mon, 5=Sat, 6=Sun
vle = vle.merge(df_vle[['id_site','activity_type']], on='id_site', how='left')

# Assessment: dung date_submitted (ngay sinh vien submit thuc te) lam moc thoi gian
# Match ban goc: chi giu assessment co deadline <= CUTOFF_DAY (safe_assessments)
sa = df_studentAssessment.merge(
    df_assessments[['id_assessment','code_module','code_presentation','date','weight','assessment_type']],
    on='id_assessment', how='inner'
)
sa['date_assessment'] = pd.to_numeric(sa['date'], errors='coerce')
sa = sa[
    (sa['assessment_type'] != 'Exam') &
    sa['date_assessment'].notna() &
    (sa['date_assessment'] <= CUTOFF_DAY)
].copy()
sa['date_submitted'] = pd.to_numeric(sa['date_submitted'], errors='coerce')
sa = sa.dropna(subset=['date_submitted'])
sa = sa[(sa['date_submitted'] >= 0) & (sa['date_submitted'] <= CUTOFF_DAY)].copy()
sa['week_submitted'] = sa['date_submitted'].astype(int) // WEEK_LEN
sa['days_before_deadline'] = sa['date_assessment'] - sa['date_submitted']

# Total assessments planned per module-presentation (cho submission rate denominator)
df_assessments['date_num'] = pd.to_numeric(df_assessments['date'], errors='coerce')
asmt_safe = df_assessments[
    df_assessments['date_num'].notna() &
    (df_assessments['date_num'] <= CUTOFF_DAY) &
    (df_assessments['assessment_type'] != 'Exam')
].copy()
asmt_safe['deadline_week'] = (asmt_safe['date_num'].astype(int) // WEEK_LEN)

print('vle (filtered+weeked) shape:', vle.shape)
print('sa  (filtered+weeked) shape:', sa.shape)
print('asmt_safe (deadlines) shape:', asmt_safe.shape)

vle (filtered+weeked) shape: (7795052, 9)
sa  (filtered+weeked) shape: (108206, 13)
asmt_safe (deadlines) shape: (113, 8)


In [6]:
# 2.2 Per-week aggregation cua VLE — RAW WEEKLY (chua cumulative)

attempt_keys = ['id_student','code_module','code_presentation']

# Total clicks + days per week
weekly_basic = vle.groupby(attempt_keys + ['week']).agg(
    clicks_this_week        = ('sum_click', 'sum'),
    active_days_this_week   = ('date', 'nunique'),
    max_daily_clicks_week   = ('sum_click', 'max'),
    median_daily_clicks_week= ('sum_click', 'median'),
    std_clicks_week         = ('sum_click', 'std'),
    last_day_in_week        = ('date', 'max'),
).reset_index()
weekly_basic['std_clicks_week'] = weekly_basic['std_clicks_week'].fillna(0)

# Activity-type breakdown per week
activity_types = ['oucontent','forumng','homepage','subpage','url','quiz']
activity_weekly = vle.pivot_table(
    index=attempt_keys + ['week'],
    columns='activity_type', values='sum_click', aggfunc='sum', fill_value=0
).reset_index()
for t in activity_types:
    if t not in activity_weekly.columns:
        activity_weekly[t] = 0
activity_weekly = activity_weekly[attempt_keys + ['week'] + activity_types]
activity_weekly = activity_weekly.rename(columns={t: f'clicks_{t}_week' for t in activity_types})

# Weekend clicks per week
weekend_weekly = vle[vle['day_of_week'].isin([5,6])].groupby(
    attempt_keys + ['week']).agg(weekend_clicks_week=('sum_click','sum')).reset_index()

print('weekly_basic:', weekly_basic.shape)
print('activity_weekly:', activity_weekly.shape)
print('weekend_weekly:', weekend_weekly.shape)

weekly_basic: (439956, 10)
activity_weekly: (439956, 10)
weekend_weekly: (252109, 5)


In [7]:
# 2.3 Per-week aggregation cua ASSESSMENT — RAW WEEKLY

# Dem so bai nop trong tuan, mean score trong tuan, weighted score trong tuan,
# avg days_before_deadline trong tuan
sa_for_agg = sa.copy()
sa_for_agg['weighted_score_contrib'] = sa_for_agg['score'] * sa_for_agg['weight']

asmt_weekly = sa_for_agg.groupby(attempt_keys + ['week_submitted']).agg(
    n_submitted_week        = ('id_assessment', 'count'),
    score_sum_week          = ('score', 'sum'),
    weight_sum_week         = ('weight', 'sum'),
    weighted_score_num_week = ('weighted_score_contrib', 'sum'),
    days_before_deadline_sum_week = ('days_before_deadline', 'sum'),
    days_before_deadline_count_week = ('days_before_deadline', 'count'),
).reset_index().rename(columns={'week_submitted':'week'})

# TMA/CMA submission counts per week (cho submission rate)
asmt_weekly_typed = sa.groupby(attempt_keys + ['week_submitted','assessment_type']).agg(
    n_sub=('id_assessment','count')
).reset_index().rename(columns={'week_submitted':'week'})

asmt_weekly_typed_pivot = asmt_weekly_typed.pivot_table(
    index=attempt_keys + ['week'], columns='assessment_type', values='n_sub', fill_value=0
).reset_index()
for t in ['TMA','CMA']:
    if t not in asmt_weekly_typed_pivot.columns:
        asmt_weekly_typed_pivot[t] = 0
asmt_weekly_typed_pivot = asmt_weekly_typed_pivot.rename(
    columns={'TMA':'n_tma_submitted_week','CMA':'n_cma_submitted_week'})

# Deadlines per (module, presentation, week): expected denominators
asmt_deadlines = asmt_safe.groupby(
    ['code_module','code_presentation','deadline_week','assessment_type']
).size().reset_index(name='n_due')
asmt_deadlines_pivot = asmt_deadlines.pivot_table(
    index=['code_module','code_presentation','deadline_week'],
    columns='assessment_type', values='n_due', fill_value=0
).reset_index().rename(columns={'deadline_week':'week'})
for t in ['TMA','CMA']:
    if t not in asmt_deadlines_pivot.columns:
        asmt_deadlines_pivot[t] = 0
asmt_deadlines_pivot = asmt_deadlines_pivot.rename(
    columns={'TMA':'n_tma_due_week','CMA':'n_cma_due_week'})

print('asmt_weekly:', asmt_weekly.shape)
print('asmt_weekly_typed_pivot:', asmt_weekly_typed_pivot.shape)
print('asmt_deadlines_pivot:', asmt_deadlines_pivot.shape)

asmt_weekly: (103766, 10)
asmt_weekly_typed_pivot: (103766, 6)
asmt_deadlines_pivot: (99, 5)


In [8]:
# 2.4 Tao panel DAY DU: (id_student, code_module, code_presentation, week) cho moi t = 0..T_MAX-1
# Mot so tuan sinh vien khong co activity -> row se co 0 (sau khi fillna)

attempts = df_static[attempt_keys].drop_duplicates().reset_index(drop=True)
weeks_df = pd.DataFrame({'week': np.arange(T_MAX)})
panel = attempts.assign(_k=1).merge(weeks_df.assign(_k=1), on='_k').drop(columns='_k')
print('Panel full grid shape:', panel.shape, f'({len(attempts)} attempts x {T_MAX} weeks)')

Panel full grid shape: (814825, 4) (32593 attempts x 25 weeks)


In [9]:
# 2.5 Merge cac weekly aggregations vao panel

panel = panel.merge(weekly_basic, on=attempt_keys + ['week'], how='left')
panel = panel.merge(activity_weekly, on=attempt_keys + ['week'], how='left')
panel = panel.merge(weekend_weekly, on=attempt_keys + ['week'], how='left')
panel = panel.merge(asmt_weekly, on=attempt_keys + ['week'], how='left')
panel = panel.merge(asmt_weekly_typed_pivot, on=attempt_keys + ['week'], how='left')
panel = panel.merge(
    asmt_deadlines_pivot,
    on=['code_module','code_presentation','week'], how='left'
)

# Cac cot weekly absolute -> fillna 0 (tuan do khong activity)
weekly_zero_cols = [
    'clicks_this_week','active_days_this_week','max_daily_clicks_week',
    'median_daily_clicks_week','std_clicks_week',
] + [f'clicks_{t}_week' for t in activity_types] + [
    'weekend_clicks_week',
    'n_submitted_week','score_sum_week','weight_sum_week',
    'weighted_score_num_week','days_before_deadline_sum_week','days_before_deadline_count_week',
    'n_tma_submitted_week','n_cma_submitted_week',
    'n_tma_due_week','n_cma_due_week',
]
for c in weekly_zero_cols:
    if c in panel.columns:
        panel[c] = panel[c].fillna(0)

# last_day_in_week: neu khong co thi de NaN (can rieng de compute days_since_last_active)
print('Panel after merge:', panel.shape)
panel.head(3)

Panel after merge: (814825, 27)


,id_student,code_module,code_presentation,week,clicks_this_week,active_days_this_week,max_daily_clicks_week,median_daily_clicks_week,std_clicks_week,last_day_in_week,...,n_submitted_week,score_sum_week,weight_sum_week,weighted_score_num_week,days_before_deadline_sum_week,days_before_deadline_count_week,n_cma_submitted_week,n_tma_submitted_week,n_cma_due_week,n_tma_due_week
0,11391,AAA,2013J,0,183.0,4.0,76.0,2.0,13.809916,6.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11391,AAA,2013J,1,20.0,1.0,16.0,1.5,7.348469,9.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,11391,AAA,2013J,2,100.0,2.0,27.0,5.0,6.618876,18.0,...,1.0,78.0,10.0,780.0,1.0,1.0,0.0,1.0,0.0,1.0


In [10]:
# 2.6 Tinh cac CUMULATIVE / CAUSAL features qua time
# Tat ca dung .groupby(attempt_keys).cumsum() / expanding() de chi nhin ve qua khu

panel = panel.sort_values(attempt_keys + ['week']).reset_index(drop=True)
gb = panel.groupby(attempt_keys)

# 2.6.a Cumulative clicks/days
panel['cum_clicks']       = gb['clicks_this_week'].cumsum()
panel['cum_active_days']  = gb['active_days_this_week'].cumsum()

# 2.6.b Active rate so far: cum_active_days / total_days_so_far
panel['days_so_far'] = (panel['week'] + 1) * WEEK_LEN   # day budget toi het tuan t
panel['active_rate_so_far'] = panel['cum_active_days'] / panel['days_so_far']

# 2.6.c Clicks per active day (cumulative)
panel['clicks_per_active_day_cum'] = np.where(
    panel['cum_active_days'] > 0,
    panel['cum_clicks'] / panel['cum_active_days'].clip(lower=1),
    0
)

# 2.6.d Days since last active (tai cuoi tuan t)
panel['last_active_day_so_far'] = gb['last_day_in_week'].ffill()
end_of_week_day = (panel['week'] + 1) * WEEK_LEN - 1
panel['days_since_last_active'] = np.where(
    panel['last_active_day_so_far'].notna(),
    end_of_week_day - panel['last_active_day_so_far'],
    end_of_week_day + 1
)

# 2.6.e Active span so far = last_active - first_active
panel['_first_active_day_so_far'] = gb['last_day_in_week'].transform(
    lambda s: s.expanding().min()
)
panel['active_span_so_far'] = (
    panel['last_active_day_so_far'].fillna(0) - panel['_first_active_day_so_far'].fillna(0)
)
panel = panel.drop(columns=['_first_active_day_so_far'])

print('Cumulative core features done.')
panel[['id_student','week','clicks_this_week','cum_clicks',
       'active_rate_so_far','days_since_last_active']].head(10)

Cumulative core features done.


,id_student,week,clicks_this_week,cum_clicks,active_rate_so_far,days_since_last_active
0,3733,0,0.0,0.0,0.0,7.0
1,3733,1,0.0,0.0,0.0,14.0
2,3733,2,0.0,0.0,0.0,21.0
3,3733,3,0.0,0.0,0.0,28.0
4,3733,4,0.0,0.0,0.0,35.0
5,3733,5,0.0,0.0,0.0,42.0
6,3733,6,0.0,0.0,0.0,49.0
7,3733,7,0.0,0.0,0.0,56.0
8,3733,8,0.0,0.0,0.0,63.0
9,3733,9,0.0,0.0,0.0,70.0


In [11]:
# 2.7 Activity-type ratios (cumulative)
for t in activity_types:
    panel[f'cum_clicks_{t}'] = gb[f'clicks_{t}_week'].cumsum()
    panel[f'ratio_{t}_so_far'] = np.where(
        panel['cum_clicks'] > 0,
        panel[f'cum_clicks_{t}'] / panel['cum_clicks'].clip(lower=1),
        0
    )

# Weekend ratio (cumulative)
panel['cum_weekend_clicks'] = gb['weekend_clicks_week'].cumsum()
panel['weekend_ratio_so_far'] = np.where(
    panel['cum_clicks'] > 0,
    panel['cum_weekend_clicks'] / panel['cum_clicks'].clip(lower=1),
    0
)

print('Activity-type ratios + weekend ratio done.')

Activity-type ratios + weekend ratio done.


In [12]:
# 2.8 Recent slope (window 4 tuan gan nhat) — proxy cho click_slope causal
from scipy.stats import linregress

def rolling_slope_r2(series, window=4):
    out_slope = np.zeros(len(series))
    out_r2 = np.zeros(len(series))
    vals = series.values
    for i in range(len(vals)):
        lo = max(0, i - window + 1)
        seg = vals[lo:i+1]
        if len(seg) >= 2 and np.std(seg) > 0 and np.sum(seg) > 0:
            try:
                r = linregress(np.arange(len(seg)), seg)
                out_slope[i] = r.slope
                out_r2[i] = r.rvalue ** 2
            except Exception:
                pass
    return out_slope, out_r2

slopes = np.zeros(len(panel))
r2s    = np.zeros(len(panel))
for keys, idx in tqdm(gb.indices.items(), desc='Slope per attempt'):
    s, r = rolling_slope_r2(panel.loc[idx, 'clicks_this_week'], window=4)
    slopes[idx] = s
    r2s[idx] = r

panel['click_slope_recent']    = slopes
panel['click_slope_r2_recent'] = r2s
print('Recent slope done.')

Slope per attempt: 100%|██████████| 32593/32593 [05:27<00:00, 99.52it/s]

Recent slope done.


In [13]:
# 2.9 Activity density + gap features (causal)
panel['activity_density_so_far'] = np.where(
    panel['active_span_so_far'] > 0,
    panel['cum_active_days'] / panel['active_span_so_far'].clip(lower=1),
    np.where(panel['cum_active_days'] >= 1, 1.0, 0.0)
)

# Gap features can list active days den tuan t -> tinh iteratively
vle_days = vle.groupby(attempt_keys)['date'].apply(lambda x: sorted(set(x.tolist()))).to_dict()

gap_std_arr  = np.zeros(len(panel))
gap_trend_arr= np.zeros(len(panel))

panel_groups = panel.groupby(attempt_keys, sort=False).indices
for keys, idx_list in tqdm(panel_groups.items(), desc='Gap features'):
    days_all = vle_days.get(keys, [])
    sub = panel.loc[idx_list].sort_values('week')
    for row_idx, week_val in zip(sub.index, sub['week'].values):
        eow = (week_val + 1) * WEEK_LEN - 1
        days_so_far = [d for d in days_all if d <= eow]
        if len(days_so_far) >= 2:
            gaps = np.diff(days_so_far)
            gap_std_arr[row_idx]  = float(np.std(gaps, ddof=0))
            if len(gaps) >= 3:
                try:
                    slope, _ = np.polyfit(np.arange(len(gaps)), gaps, 1)
                    gap_trend_arr[row_idx] = slope
                except Exception:
                    pass
panel['gap_std_so_far']   = gap_std_arr
panel['gap_trend_so_far'] = gap_trend_arr
print('Gap features done.')

Gap features: 100%|██████████| 32593/32593 [02:09<00:00, 251.42it/s]

Gap features done.


In [14]:
# 2.10 Cumulative assessment features
panel['cum_n_submitted']         = gb['n_submitted_week'].cumsum()
panel['cum_score_sum']           = gb['score_sum_week'].cumsum()
panel['cum_weight_sum']          = gb['weight_sum_week'].cumsum()
panel['cum_weighted_score_num']  = gb['weighted_score_num_week'].cumsum()
panel['cum_dbd_sum']             = gb['days_before_deadline_sum_week'].cumsum()
panel['cum_dbd_count']           = gb['days_before_deadline_count_week'].cumsum()

panel['cum_n_tma_submitted']     = gb['n_tma_submitted_week'].cumsum()
panel['cum_n_cma_submitted']     = gb['n_cma_submitted_week'].cumsum()
panel['cum_n_tma_due']           = gb['n_tma_due_week'].cumsum()
panel['cum_n_cma_due']           = gb['n_cma_due_week'].cumsum()

# Derived (causal)
panel['mean_score_so_far'] = np.where(
    panel['cum_n_submitted'] > 0,
    panel['cum_score_sum'] / panel['cum_n_submitted'].clip(lower=1),
    0
)
panel['weighted_score_so_far'] = np.where(
    panel['cum_weight_sum'] > 0,
    panel['cum_weighted_score_num'] / panel['cum_weight_sum'].clip(lower=1),
    0
)
panel['has_weighted_score_so_far'] = (panel['weighted_score_so_far'] > 0).astype(int)
panel['avg_days_before_deadline_so_far'] = np.where(
    panel['cum_dbd_count'] > 0,
    panel['cum_dbd_sum'] / panel['cum_dbd_count'].clip(lower=1),
    0
)
panel['tma_submission_rate_so_far'] = np.where(
    panel['cum_n_tma_due'] > 0,
    panel['cum_n_tma_submitted'] / panel['cum_n_tma_due'].clip(lower=1),
    -1
)
panel['cma_submission_rate_so_far'] = np.where(
    panel['cum_n_cma_due'] > 0,
    panel['cum_n_cma_submitted'] / panel['cum_n_cma_due'].clip(lower=1),
    -1
)
panel['n_missing_so_far'] = (
    (panel['cum_n_tma_due'] - panel['cum_n_tma_submitted']).clip(lower=0) +
    (panel['cum_n_cma_due'] - panel['cum_n_cma_submitted']).clip(lower=0)
)
print('Assessment cumulative features done.')

Assessment cumulative features done.


## 3. Feature lists & Encoding

In [15]:
# 3.1 Dynamic feature list cho moi timestep
DYNAMIC_FEATURES = [
    # Raw weekly
    'clicks_this_week','active_days_this_week',
    'max_daily_clicks_week','median_daily_clicks_week','std_clicks_week',
    'weekend_clicks_week','n_submitted_week',
    # Cumulative / so-far
    'cum_clicks','cum_active_days','active_rate_so_far',
    'clicks_per_active_day_cum',
    'days_since_last_active','active_span_so_far',
    'click_slope_recent','click_slope_r2_recent',
    'activity_density_so_far','gap_std_so_far','gap_trend_so_far',
    'weekend_ratio_so_far',
] + [f'ratio_{t}_so_far' for t in activity_types] + [
    'mean_score_so_far','weighted_score_so_far','has_weighted_score_so_far',
    'avg_days_before_deadline_so_far',
    'tma_submission_rate_so_far','cma_submission_rate_so_far',
    'n_missing_so_far','cum_n_submitted',
]
print(f'So dynamic features per timestep: {len(DYNAMIC_FEATURES)}')
print(DYNAMIC_FEATURES)

So dynamic features per timestep: 33
['clicks_this_week', 'active_days_this_week', 'max_daily_clicks_week', 'median_daily_clicks_week', 'std_clicks_week', 'weekend_clicks_week', 'n_submitted_week', 'cum_clicks', 'cum_active_days', 'active_rate_so_far', 'clicks_per_active_day_cum', 'days_since_last_active', 'active_span_so_far', 'click_slope_recent', 'click_slope_r2_recent', 'activity_density_so_far', 'gap_std_so_far', 'gap_trend_so_far', 'weekend_ratio_so_far', 'ratio_oucontent_so_far', 'ratio_forumng_so_far', 'ratio_homepage_so_far', 'ratio_subpage_so_far', 'ratio_url_so_far', 'ratio_quiz_so_far', 'mean_score_so_far', 'weighted_score_so_far', 'has_weighted_score_so_far', 'avg_days_before_deadline_so_far', 'tma_submission_rate_so_far', 'cma_submission_rate_so_far', 'n_missing_so_far', 'cum_n_submitted']


In [16]:
# 3.2 Encode static features (giong tabular notebook goc)
from sklearn.preprocessing import LabelEncoder

education_order = ['No Formal quals','Lower Than A Level','A Level or Equivalent',
                   'HE Qualification','Post Graduate Qualification']
age_order = ['0-35','35-55','55<=']
imd_order = ['0-10%','10-20%','20-30%','30-40%','40-50%',
             '50-60%','60-70%','70-80%','80-90%','90-100%','Unknown']

df_static['highest_education_encoded'] = df_static['highest_education'].map(
    {v:i for i,v in enumerate(education_order)}).fillna(2)
df_static['age_band_encoded'] = df_static['age_band'].map(
    {v:i for i,v in enumerate(age_order)}).fillna(0)
df_static['imd_band_encoded'] = df_static['imd_band'].map(
    {v:i for i,v in enumerate(imd_order)}).fillna(5)
df_static['semester_enc']     = (df_static['semester'] == 'B').astype(int)
df_static['gender_encoded']   = df_static['gender'].map({'M':0,'F':1}).fillna(2).astype(int)
df_static['disability_encoded']= df_static['disability'].map({'N':0,'Y':1}).fillna(2).astype(int)
print('Categorical encoding done.')

Categorical encoding done.


In [17]:
# 3.3 OOT split: hoc ky cuoi cung = test
presentations = sorted(df_static['code_presentation'].unique())
train_semesters = presentations[:-1]
test_semesters  = [presentations[-1]]
print(f'Train cohorts: {train_semesters}')
print(f'Test cohort (OOT): {test_semesters}')

is_train = df_static['code_presentation'].isin(train_semesters)
is_test  = df_static['code_presentation'].isin(test_semesters)
df_static_train = df_static[is_train].copy().reset_index(drop=True)
df_static_test  = df_static[is_test].copy().reset_index(drop=True)
print(f'Train attempts: {len(df_static_train)} | Test attempts: {len(df_static_test)}')

Train cohorts: ['2013B', '2013J', '2014B']
Test cohort (OOT): ['2014J']
Train attempts: 21333 | Test attempts: 11260


In [18]:
# 3.4 Module-semester encoder (fit on train, safe transform test)
le_module = LabelEncoder()
df_static_train['module_semester_encoded'] = le_module.fit_transform(df_static_train['module_semester'])
known = set(le_module.classes_)
df_static_test['module_semester_encoded'] = df_static_test['module_semester'].apply(
    lambda x: le_module.transform([x])[0] if x in known else -1
)

# Region dummies
region_train = pd.get_dummies(df_static_train['region'], prefix='region', drop_first=True).astype(int)
region_test  = pd.get_dummies(df_static_test['region'],  prefix='region', drop_first=True).astype(int)
region_test  = region_test.reindex(columns=region_train.columns, fill_value=0)
df_static_train = pd.concat([df_static_train, region_train], axis=1)
df_static_test  = pd.concat([df_static_test,  region_test],  axis=1)

STATIC_FEATURES = [
    'highest_education_encoded','age_band_encoded','imd_band_encoded',
    'semester_enc','gender_encoded','disability_encoded',
    'module_semester_encoded',
    'studied_credits','num_of_prev_attempts','date_registration',
    'pre_course_clicks','pre_course_active_days','has_pre_course_activity',
] + list(region_train.columns)

print(f'Static features: {len(STATIC_FEATURES)}')
print(STATIC_FEATURES)

Static features: 25
['highest_education_encoded', 'age_band_encoded', 'imd_band_encoded', 'semester_enc', 'gender_encoded', 'disability_encoded', 'module_semester_encoded', 'studied_credits', 'num_of_prev_attempts', 'date_registration', 'pre_course_clicks', 'pre_course_active_days', 'has_pre_course_activity', 'region_East Midlands Region', 'region_Ireland', 'region_London Region', 'region_North Region', 'region_North Western Region', 'region_Scotland', 'region_South East Region', 'region_South Region', 'region_South West Region', 'region_Wales', 'region_West Midlands Region', 'region_Yorkshire Region']


## 4. Final Export for ML (Snapshots)

Thay vì tạo chuỗi tuần tự (Tensor) như mô hình Deep Learning, đối với mô hình dạng cây (XGBoost/LightGBM), chúng ta sẽ trích xuất các bản chụp (Snapshots) tại các tuần cắt (cut-off) cụ thể: `4, 8, 12, 16, 20`.

**Xử lý `fillna(0)`:**
Đối với số đếm (số click, số ngày truy cập), điền `0` là hoàn toàn chính xác (không truy cập = 0). Tuy nhiên, với các biến số mang tính tỷ lệ hoặc trung bình (VD: `mean_score`, `active_rate`), nếu điền `0` sẽ làm sai lệch bản chất (bị hiểu lầm là học sinh thi được 0 điểm). Do đó, với các biến này, ta sẽ điền `np.nan` để thuật toán Cây tự động đưa chúng vào nhánh Missing Values (hiểu là: chưa thi).

In [19]:
import os
import json
import numpy as np

os.makedirs(OUTPUT_PATH, exist_ok=True)

# Merge panel voi df_static de lay Target va thong tin tinh
df_static_updated = pd.concat([df_static_train, df_static_test], axis=0)
full_df = panel.merge(df_static_updated, on=attempt_keys, how='inner')

cut_off_weeks = [4, 8, 12, 16, 20, 24]

# Tim cac cot mang tinh ty le/trung binh de replace 0 thanh NaN
ratio_mean_cols = [c for c in DYNAMIC_FEATURES if 'ratio' in c or 'mean' in c or 'avg' in c or 'rate' in c]
print('Cac cot se duoc ap dung NaN thay vi 0:', ratio_mean_cols)

for w in cut_off_weeks:
    # Lay chinh xac dong du lieu tai tuan w (khong the bi leak tuong lai do data duoc mask den w)
    snap = full_df[full_df['week'] == w].copy()
    
    # Xu ly Missing Values cho mo hinh Cay
    for c in ratio_mean_cols:
        snap[c] = snap[c].replace(0.0, np.nan)
        
    cols = attempt_keys + ['target'] + DYNAMIC_FEATURES + STATIC_FEATURES
    snap_final = snap[cols].reset_index(drop=True)
    
    # Chia Train/Test dung Out-Of-Time (OOT)
    train_w = snap_final[snap_final['code_presentation'].isin(train_semesters)]
    test_w  = snap_final[snap_final['code_presentation'].isin(test_semesters)]
    
    # Export
    train_w.to_csv(OUTPUT_PATH + f'df_train_week_{w}.csv', index=False)
    test_w.to_csv(OUTPUT_PATH + f'df_test_week_{w}.csv', index=False)
    print(f'Da luu Snapshot Tuan {w} - Train: {train_w.shape}, Test: {test_w.shape}')

# Luu danh sach features de model de dang su dung
with open(OUTPUT_PATH + 'feature_cols.json', 'w') as f:
    json.dump(DYNAMIC_FEATURES + STATIC_FEATURES, f, indent=2)
print('Data Prep cho ML hoan tat!')


Cac cot se duoc ap dung NaN thay vi 0: ['active_rate_so_far', 'weekend_ratio_so_far', 'ratio_oucontent_so_far', 'ratio_forumng_so_far', 'ratio_homepage_so_far', 'ratio_subpage_so_far', 'ratio_url_so_far', 'ratio_quiz_so_far', 'mean_score_so_far', 'avg_days_before_deadline_so_far', 'tma_submission_rate_so_far', 'cma_submission_rate_so_far']
Da luu Snapshot Tuan 4 - Train: (21333, 62), Test: (11260, 62)
Da luu Snapshot Tuan 8 - Train: (21333, 62), Test: (11260, 62)
Da luu Snapshot Tuan 12 - Train: (21333, 62), Test: (11260, 62)
Da luu Snapshot Tuan 16 - Train: (21333, 62), Test: (11260, 62)
Da luu Snapshot Tuan 20 - Train: (21333, 62), Test: (11260, 62)
Da luu Snapshot Tuan 24 - Train: (21333, 62), Test: (11260, 62)
Data Prep cho ML hoan tat!
